# Python primer IV 

## Topics to cover

- working with regular expressions
- various `re` functions (`re.search`, `re.finditer`, `re.findall`, `re.sub`, etc. )
- counting and storing matches
- using regexpressions for general data processing


## Useful materials
- Haddock and Dunn chapters and 10, bit of 11 
- regular expression cheat sheet (python-regular-expressions-cheat-sheet-1.pdf)
- [regular expression tester](https://regex101.com)
- [python for biologists regular expression lesson](https://pythonforbiologists.com/tutorial/regex.html)
- [pythonguru regular expression tutorial](https://thepythonguru.com/python-regular-expression/)



## Introduction to regular expressions
We have already used regular expressions with Unix (e.g., `grep`, `sed`, `tr`, `awk`) and Python (`str.count()`) tools that allow pattern recognition as a basis for completing some task or file manipulation. However, we were using these tools mostly to search for fixed patterns. While working with biological data, and really any other form of big data, we will encounter many problems that will require the ability to match flexible patterns. 

Because pattern matching is a common task for working with big data, there is a standard set of tools in Python (and other languages) for dealing with them: regular expressions. Regular expressions are statements in code designed to match text with certain characteristics (ranging from exact matches to extremely fuzzy ones), are extremely useful in biology and across data science in general. This week we will start to learn about how to match specific and increasingly flexible patterns in data, as well as how to track, store, and manipulate the portions of strings that we wish to extract. Moving forward, you are likely to use regular expressions in much of the code you write in Python.

We will start working with regular expressions using the `re` module. As hinted previously with `sys`, libraries will play a major role in enabling your python code. We will cover that in more detail in a few weeks. For now, lets cover the basics of matching regular expressions with `re`.

To use functionality in the `re` library in your code, you will add a line near the top of your scripts that uses the `import` function. 


In [4]:
import re

The `re` module in Python provides a small set of functions that handle most regex tasks. Below is a tabular summary before we move into detailed examples and focus on `re.search`,`re.finditer`, `re.findall`, `re.sub`, and more.

| **Function** | **Typical Use** | **Example Scenario** |
|---------------|------------------|-----------------------|
| `re.search()` | Finds the *first* match of a pattern anywhere in a string. Returns a **match object** or `None`. | Use inside `if` statements to test whether a sequence contains a stop codon (`if re.search(r"TAA|TAG|TGA", seq):`). |
| `re.match()` | Like `search()`, but only matches if the pattern occurs at the **beginning** of the string. | Detect if a FASTA line starts with `>` (e.g., `if re.match(r"^>", line):`). |
| `re.findall()` | Returns a **list** of all matches (as plain text). | Extract all accession IDs or codons from a sequence. |
| `re.finditer()` | Like `findall()`, but returns **iterators** that include position information (`.start()`, `.end()`). | Useful for locating where motifs or codons occur along a sequence. |
| `re.sub()` | Replaces all instances of a pattern with new text. | Replace `T` with `U` in an RNA conversion, or remove unwanted symbols from headers. |
| `re.split()` | Splits a string using a regex pattern as the delimiter. | Break up GFF or CSV lines by tabs, commas, or multiple spaces. |
| `re.compile()` | Prepares (or “compiles”) a pattern for reuse. Speeds up repeated searches. | Ideal when applying the same pattern inside a loop scanning many sequences. |

---

### When to Use Regular Expressions in Bioinformatics and Data Science

Regular expressions are one of the most versatile text-processing tools in programming. They help you detect, extract, transform, and validate patterns in any kind of data, filenames, logs, or data stored in large collections of text.  

Below are some common situations where regular expressions are invaluable.

#### In Bioinformatics
- **Sequence validation:** Ensure that DNA or RNA strings contain only valid bases (e.g., `A`, `T`, `G`, `C`, `U`, or `N`).
- **Motif or pattern detection:** Search for start codons, restriction enzyme sites, or conserved motifs across sequences.
- **Parsing annotation files:** Extract gene IDs, positions, or feature types from GFF, BED, or GenBank records.
- **Cleaning FASTA headers:** Remove or simplify complex text in sequence headers to standardize file formats.
- **Data wrangling for downstream tools:** Convert delimiters, rename files, or extract sample identifiers automatically.
- **Quality control:** Detect missing sequence data, bad line endings, or formatting errors before running large analyses.
- **Batch renaming and metadata extraction:** Identify and standardize filenames like `sample_A01.fastq.gz` into structured metadata.

#### In data science and general programming
- **Data cleaning:** Identify or remove unwanted symbols, multiple spaces, or inconsistent formatting in CSV or text files.
- **Text mining:** Extract specific keywords, numbers, or identifiers from unstructured text (e.g., gene names, DOIs, accession numbers).
- **Validation tasks:** Confirm that emails, URLs, or numeric fields are properly formatted before loading data.
- **Log file analysis:** Parse structured text logs from servers or HPC jobs to summarize errors, runtime, or memory usage.
- **Web scraping and automation:** Identify text or data patterns within HTML, JSON, or API responses.
- **Pattern-based grouping:** Detect repeated sequences, trends, or structured tokens for machine learning preprocessing.
- **Coding efficiency:** Replace long chains of `.split()` and `.replace()` with compact, readable one line solutions.

#### Regular expressions are generally useful whenever:
- Your data is semi-structured** (not perfectly tabular)  
- You need to search for or extract patterns rather than fixed strings  
- You need to clean, validate, or transform text quickly and reproducibly  

In other words, regular expressions allow you to automate what would be tedious “find and replace” work — and to scale that capability to entire datasets or directories.


## `re.search()`

In general, `re` requires you to specify a pattern to match, followed by a string to match the pattern in. There are numerous functions built into `re`, but lets start here with `re.search`, which returns an object with information about the match and true/false based on whether the match occurs in the string or not. The general idea is as follows:

    re.search(pattern, string)

We can search a specified string, using an `if` statement as an example of how to control our code based on the presence or absence of matches:

In [5]:
Seq = "ATCGGGGCCTAGAAT"
if re.search("TAG", Seq):
    print("Stop codon (TAG) found.\n")
else:
    print("specified codon not found")

Stop codon (TAG) found.



Note if we use just the `re.search` command in our jupyter-notebook, it will return the data object pertaining to the stored details of the match. This can be stored for use as needed. **Note** `re.search` is fast because it only returns/stores the first match in a string.

In [6]:
m1=re.search("TAG", Seq)
print(m1)

<re.Match object; span=(9, 12), match='TAG'>


The `^` character can be used to anchor the pattern at the beginning of the string, and the `$` anchors the pattern at the end. Note that `^` used in a character class list (characters enclosed in brackets), such as [^ATCG] means something different (in this case it would any character that is NOT ATCG). We will cover this further below.

In [4]:
Seq = "ATCGGGGCCTAGAAT"
if re.search("^A", Seq):
    print("Seq begins with A.\n")

Seq = "ATCGGGGCCTAGAAT"
if re.search("T$", Seq):
    print("Seq ends with T.\n")

Seq begins with A.

Seq ends with T.



We can use **alternation** (pattern A OR pattern B) to increase flexibility of  expressions.

In [5]:
Seq = "ATCGGGGCCTAGAAT"
if re.search("CC(C|T)", Seq):
    print("Seq has 'CCC' or 'CCT'.\n")

Seq has 'CCC' or 'CCT'.



Matches can be saved to variables:

In [7]:
Seq = 'ATCGGGGGGATCGGGATC'
Seqmatch=re.search('ATC', Seq)
print(Seqmatch)

<re.Match object; span=(0, 3), match='ATC'>


Matches can be extracted as below, although we will cover capturing and storing matches in more detail below with other `re` functions  (`re.search` only stores first match).

In [8]:
Seq = 'ATCGGGGGGATCGGGATC'
match= re.search('ATC', Seq)
print(match.group()) 

ATC


Match objects have multiple elements, which can be individually extracted:

In [9]:
Seq = 'ATCGGGGGGATCGGGATC'
Seqmatch=re.search('ATC', Seq)
print(Seqmatch.start())  
print(Seqmatch.group()) 
print(Seqmatch.end()) 

0
ATC
3


Note that above, `re.search` only stores one ATC, even though there are 3 in Seq. Thats because `re.search` only makes one match. `re.findall` and `re.finditer` can be used to find, track, and store multiple matches. More on these below.



### Note: why you should generally or usually use raw string (`r""`) syntax of regular expressions in python

When writing regular expressions in Python, you’ll often see patterns that include backslashes — for example, `\d` (any digit), `\s` (whitespace), or `\w` (alphanumeric).  
However, in normal Python strings, the backslash `\` is a **special escape character**, used for things like:
- `\n` → newline  
- `\t` → tab  
- `\\` → a literal backslash

For example:

In [27]:
print("\t", "that was a tab")   # Python inserts a tab
print(r"\t")  # Python prints the characters '\' and 't'

	 that was a tab
\t


Because of this, if you write:
```python
"\d"
```
Python interprets it as just "d", not as the regex \d. To safeguard against this getting weird, you have two options:

Option 1: Escape the backslash:

In [25]:
re.search("\\d+", "abc123")

<re.Match object; span=(3, 4), match='1'>

**Option 2 (Preferred)**: Use a raw string

In [24]:
re.search(r"\d+", "abc123")

<re.Match object; span=(3, 4), match='1'>

Adding the r in front of the quotes tells Python to treat the string literatlly and dont interpret backslashes.”

This means the regex engine receives exactly what you typed: \d+, not "d+".

It is generally a good idea to always use raw strings (r"pattern") when writing regular expressions in Python becauese it easier to read, safer, and can prevent hidden bugs.

Example:

In [9]:
if re.search(r"[ATGC]{3}", "ATGCCG"):
    print("Found a codon")


Found a codon


In [10]:
Seq = 'ATCGGGGGGATCGGGATC'
Seqmatch=re.search(r'ATC', Seq)
print(Seqmatch)

<re.Match object; span=(0, 3), match='ATC'>


It will matter in strings that have special characters, such as '\n', which is a linux line ending. When preceded in a match statement by 'r', backslashes are not treated differently than any other charater. So r"\n" is a two-character string containing '\' and 'n', while "\n" is a one-character string containing a newline. Usually patterns will be expressed in Python code using this raw string notation, so we will mostly use in what follows.


## Basic syntax for regular expression matching

Syntax for regular expressions is mostly consistent across many languages, including Unix, Perl, and Python. The regular expression cheat sheet posted with the course materials will be helpful here; I suggest keeping that or similar cheat sheet close by. The table below illustrates some of the most commonly used expressions



| Expression | Meaning |
|---------- | ---------- |
|\d  | single digit character |
|\d+  | One or more digits |
|\d*  | 0 or more digits |
|\w  | word character |
|\W | non word character |
|\s | whitespace character |
|\S | non-whitespace character |
|*  | anything|
|.| any non-whitespace character|
|^ | beginning of string anchor |
|\$ | end of string anchor|
|\d\{3,\} | 3 or more consecutive digits|
|[a-z]| any lower case letter|
|[A-Z]| any upper case letter|
|[^ATCG]| any character other than A, T, C, G|
|\\$| matches "$", special characters 'escaped' with `\`|
|\\@| matches "@", special characters 'escaped' with `\`|

## More on special characters

Special characters require a `\` escape in regular expressions. Thus, matching '&' in a string of text requires an expresssion such as "\\&". In addition, `\` is used to indicate certain special characters such as line endings, tabs, and other characters listed above. Examples below.

So in the string below, specifying search with "\\@" will produce a match, while using "@" will not. Try it out to convince yourself.


In [12]:
Seq = '*ATCGGGGGGATCGGGATC'
re.search(r"\*", Seq)   # returns a match
#re.search(r"*", Seq)   # does not return a match

<re.Match object; span=(0, 1), match='*'>


| Expression | Translation |
|---------- | ---------- |
|\\' |	single quote 	|
|\\* |	* 	|
|\\+ |	+ 	|
|\\? |	? 	|
|\n |	New Line 	|
|\r |	Carriage Return |	
| \t |	Tab 	|
| \b |	Backspace |	
| \f |	Form Feed |	



Example usage. If you run the same searches below without "\\" you will see that no match will be returned.

In [29]:
X = "COD * ? + \n"
if re.search(r"\+", X):  
    print("Matched +")
if re.search(r"\*", X):  
    print("Matched *")   
if re.search(r"\?", X):  
    print("Matched ?")
    

Matched +
Matched *
Matched ?


## `re.finditer()`

`re.search` is useful for finding out if a pattern exists in a string or a line of data, and it can store a single match, but its usefulness is limited to a single match. `re.finditer` works like `re.findall`, but instead of returning a list of matches, it returns an iterator of match objects, each containing detailed information about the match — including its text, start, and end positions.

It’s especially useful when you need to locate where patterns occur in a sequence or text

The below expression matches to characters that are NOT A, T, C or G, and the for loop is used to print each match and its starting position. Note that the "^" character when used inside a character class means "NOT" instead of serving as an anchor.


In [30]:
Seq = "CGCTCNTAGATGCGCRATGACTGCAYTGC" 

matches = re.finditer(r"[^ATGC]", Seq) # regex means any character that is NOT A, C, T, or G. Here that would be N, R, Y
for m in matches: 
    base = m.group() 
    pos  = m.start() 
    print(base + "," + str(pos))

N,5
R,15
Y,25


## Matching repetitive patterns.

We will cover this in more detail below with `re.findall`, but here show a few examples using `re.finditer`.

To extract AT repeated **2 or more**  times, and to loop through the list of matches (which are stored as m.group() elements):

In [35]:
SSR = 'ATATATGGGCGATATATATCCATATC'
matches=re.finditer(r"(AT){2,}", SSR)
for m in matches:
    print(m.group())
    print(m.start())
    


ATATAT
0
ATATATAT
11
ATAT
21


To extract ATC repeated **3 or more**  times:


In [31]:
SSR = 'ATCATCATCATCATGGGCGATATATATCCATATC'
matches=re.finditer(r"(ATC){3,}", SSR)
for m in matches:
    print(m.group())
    print(m.start())

ATCATCATCATC
0


## `re.findall()`

`re.findall` can process multiple matches and returns the text of the matches themselves as strings. Thus, rather than returning a list of match objects, this returns a list of strings. Useful for capturing or counting matches, and produces lists that are easy to work with..

In each below use of `re.findall` matches are stored as lists. We will talk more about capturing matches below.


In [22]:
ID = "CO_BC_13292929 0 1 2 0 2 1 1 "
print(re.findall(r"[A-Z]+",ID)) # returns['CO', 'BC']
print(re.findall(r"\d{3,}",ID)) # returns['13292929']

match = re.findall(r"\s\d",ID) 
for m in match:
    print(m)

['CO', 'BC']
['13292929']
 0
 1
 2
 0
 2
 1
 1


### Storing matches of repetitive patterns. Similar to above with `re.finditer`, here is an alternative way that usefully puts matches directly into a list.

To extract AT repeated **2 or more**  times, and to loop through the list of matches:

In [23]:
SSR = 'ATATATGGGCGATATATATCCATATC'
matches=re.findall(r"(AT){2,}", SSR)
for m in matches:
    print(m)


AT
AT
AT


Take note of the parentheses surrounding the base pattern (AT) above, and especially what the print statement above produced. Specifically, while the number of matches stored in the list is correct (and such use will be useful for this weeks programming task), the pattern saved as a group was the base AT, not the entire repeating region.

With a slight change (the addition of ()s around the entire expression), the exact repeating match and the base are stored in a slightly more complex list. Try the below code, and pay particular attention to how parentheses are organized and the printed output.

In [24]:
SSR = 'ATATATGGGCGATATATATCCATATC'
matches=re.findall(r"((AT){2,})", SSR)
for m in matches:
    print(m)

('ATATAT', 'AT')
('ATATATAT', 'AT')
('ATAT', 'AT')


To extract ATC repeated **3 or more**  times:


In [21]:
SSR = 'ATCATCATCATCATGGGCGATATATATCCATATC'
matches=re.findall(r"((ATC){3,})", SSR)
for m in matches:
    print(m)

('ATCATCATCATC', 'ATC')


Or if we wanted to extract regions of DNA that only have T and/or A for 5 consecutive bases:

In [26]:
SSR = 'ATAAAATCATCATATTTATGGGCGATATATATCCATATC'
matches=re.findall("[AT]{5,}", SSR)
for m in matches:
    print(m)

ATAAAAT
ATATTTAT
ATATATAT



## Counting the number of matches in a string

There are multiple ways to do this. One kind of straight forward way uses `re.findall`, which will return strings of each match, in a list if there is more than one.

In [27]:
Sent = 'the dog jumped over the moon, which circled the Earth'
Result = re.findall("the", Sent) 
print(Result) # will be a list ['the', 'the', 'the']

['the', 'the', 'the']


The length of the list generated from `re.findall` gives you the number of matches.

In [28]:
len(Result)


3

Or more efficiently, the line below will do:


In [30]:
NumMatch = len(re.findall("the", Sent)) 
print(NumMatch)

3


Another example, lets say we want to know how many 'AT' repeats occur in a DNA sequence

In [31]:
Seq = "CTGCATTATATCGTACGAAATTATACGCGCGATATATATATATATAT"
len(re.findall('AT', Seq)) ## returns 13

13

Below, an expression is used to match regions with only A and/or T for 4 or more consecutive bases. The results of `re.findall` are then used in a for loop to print the length of each AT rich match.

In [32]:
Seq = "CTGCATTATACGAATTATACGCGCGATATAATACATATAT"
ATlist = re.findall("[AT]{4,}", Seq)
for Rep in ATlist:
    print(Rep, len(Rep)) 

ATTATA 6
AATTATA 7
ATATAATA 8
ATATAT 6


## Storing matches

There are numerous ways to do capture and store matches within regular expresssions. This can be an incredibly powerful way to extract information from large files of text, especially when using flexible regular expressions. 

Parentheses are used to define parts of expression matches to store.

In [32]:
X = 'the dog jumped over the moon'
re.search(r'(dog)\s(jumped)', X).group(1) # returns dog
re.search(r'(dog)\s(jumped)', X).group(2) # returns jumped


'jumped'

Another way of looking at this with more flexible expression:


In [34]:
Name = "Loxia curvirostra" 
m = re.search(r"(\w+) (\w+)", Name) 
print(m.group(1)) # prints 'Loxia'
print(re.search(r"(\w+) (\w+)", Name).group(2)) # prints 'curvirostra

Loxia
curvirostra


Storing lists of matches with `re.finditer`:

To extract AT repeated **2 or more**  times, and to loop through the list of matches:

In [37]:
SSR = 'ATATATGGGCGATATATATCCATATC'
matches=re.finditer("(AT){2,}", SSR)
for m in matches:
    print(m.group(), len(m.group()))

ATATAT 6
ATATATAT 8
ATAT 4


To extract ATC repeated **3 or more**  times:


In [38]:
SSR = 'ATCATCATCATCATGGGCGATATATATCCATATC'
matches=re.finditer("(ATC){3,}", SSR)
for m in matches:
    print(m.group(), len(m.group()))

ATCATCATCATC 12


## Splitting strings using a regular expression: `re.split`

We are accustomed to splitting strings based on delimiters such as ',', '\s', and '\t', but in reality we can split strings with any expression we like using regular expressions and `re.split`


In [39]:
Seq = "ATGGGNCTGANNTAAAGNNNNGTCCCCNNNTTTTTTT"
MotifList = re.split("N{2,}", Seq) # splitting by '2 or more Ns
print(MotifList)

['ATGGGNCTGA', 'TAAAG', 'GTCCCC', 'TTTTTTT']


Comma delmited data string:


In [40]:
Dataline = "EcoRI,Mse1,HindIII,JoeBurrow,JamarChase"
DataList = re.split(",", Dataline) # splitting by ',''
for element in DataList:
    print(len(element))

5
4
7
9
10


## `re.sub`
`re.sub` can be used to replace a string that matches a regular expression. It takes  a regular expression pattern in the first argument, a new string in the second argument, and the string to be processed in the third argument.

In [43]:
Seq = 'CO_MT_134545 0   1 0   2 1'
result = re.sub('CO_',  '',    Seq) # Deletes CO_
print("result 1: ", result)
result = re.sub(r'\s+', ' ',   Seq) # Delete extra whitespaces 
print("result 2: ", result)

Let = 'abcdefghi'
result = re.sub('abc(def)ghi', r'\1', Let) # Replace string with a part of itself
print("result 3: ", result)


result 1:  MT_134545 0   1 0   2 1
result 2:  CO_MT_134545 0 1 0 2 1
result 3:  def


## Simple example demo

Below is a script that performs a few simple substitutions on the yeast_genome.gff file we worked with earlier in the semester. Providing this here as an example workflow, or as a basis for playing with regular expression actions.

In [1]:
import sys
import re

IN = open("top100lines_yeast_genome.gff", 'r')
OUT = open("CHR1_5_ygenome.txt", 'w')

for Line in IN:
    Line = Line.strip('\n')
    RLine = re.sub('chr','CHR', Line)
    print(RLine)

if re.search('CHRI|CHRV', RLine):
    OUT.write(RLine + "\n")

CHRI	SGD	CHRomosome	1	230208	.	.	.	ID=CHRI;dbxref=NCBI:NC_001133;Name=CHRI
CHRI	SGD	repeat_region	1	62	.	-	.	ID=TEL01L-TR;Name=TEL01L-TR;Note=Terminal%20stretch%20of%20telomeric%20repeats%20on%20the%20left%20arm%20of%20Chromosome%20I;dbxref=SGD:S000028864
CHRI	SGD	telomere	1	801	.	-	.	ID=TEL01L;Name=TEL01L;Note=Telomeric%20region%20on%20the%20left%20arm%20of%20Chromosome%20I%3B%20composed%20of%20an%20X%20element%20core%20sequence%2C%20X%20element%20combinatorial%20repeats%2C%20and%20a%20short%20terminal%20stretch%20of%20telomeric%20repeats;dbxref=SGD:S000028862
CHRI	SGD	repeat_region	63	336	.	-	.	ID=TEL01L-XR;Name=TEL01L-XR;Note=Telomeric%20X%20element%20combinatorial%20Repeat%20region%20on%20the%20left%20arm%20of%20Chromosome%20I%3B%20contains%20repeats%20of%20the%20D%2C%20C%2C%20B%20and%20A%20types%2C%20as%20well%20as%20Tbf1p%20binding%20sites%3B%20formerly%20called%20SubTelomeric%20Repeats;dbxref=SGD:S000028866
CHRI	SGD	gene	335	649	.	+	.	ID=YAL069W;Name=YAL069W;Ontology_term=GO:000

## Summary of regular expressions (`re`) syntax and functions

| **Function / Symbol** | **Example Syntax (always use raw strings)** | **Description / Use Case** |
|------------------------|---------------------------------------------|-----------------------------|
| **Importing the Library** | `import re` | Loads Python’s built-in regular expression module. Must be imported before use. |
| **Search for a pattern** | `re.search(r"ATG", seq)` | Searches anywhere in the string for `"ATG"`. Returns a *match object* if found, otherwise `None`. |
| **Match at beginning only** | `re.match(r"ATG", seq)` | Similar to `search`, but only matches if pattern occurs **at the start** of the string. |
| **Find all matches** | `re.findall(r"A[AT]G", seq)` | Returns a **list** of all non-overlapping matches. |
| **Find iterator (memory-efficient)** | `re.finditer(r"A[AT]G", seq)` | Returns an iterator over all matches (useful for large files). |
| **Substitute text** | `re.sub(r"T", "U", seq)` | Replaces all matches of pattern `"T"` with `"U"`. |
| **Split by pattern** | `re.split(r"[,;\s]+", line)` | Splits a string by one or more commas, semicolons, or spaces. |
| **Compile a pattern** | `pattern = re.compile(r"[ATGC]{3}")` | Compiles a regex for reuse (faster if used repeatedly in loops). |
| **Get matched text** | `m = re.search(r"ATG", seq); m.group()` | Returns the actual substring that matched the pattern. |
| **Get match positions** | `m.start(), m.end()` | Returns starting and ending indices of the match. |
| **Case-insensitive match** | `re.search(r"atg", seq, re.IGNORECASE)` | Matches regardless of case. |
| **Multiline mode** | `re.findall(r"^>", text, re.MULTILINE)` | Makes `^` and `$` apply to **each line** in multi-line text. |



## Commonly Used Regular expression symbols and metacharacters

| **Symbol / expression** | **Meaning** | **Example** | **Matches** |
|--------------------------|-------------|--------------|--------------|
| `.` | Any single character (except newline) | `r"A.C"` | `"ATC"`, `"AAC"`, `"ACC"` |
| `^` | Start of string or line | `r"^ATG"` | `"ATG..."` only if it appears at start |
| `$` | End of string or line | `r"TAA$"` | Matches `"TAA"` only if it ends the line |
| `[ ]` | Character class | `r"[ATGC]"` | Any one of `A`, `T`, `G`, or `C` |
| `[^ ]` | Negated class | `r"[^0-9]"` | Any character **not** a digit |
| `*` | 0 or more repetitions | `r"A*"` | `""`, `"A"`, `"AA"`, `"AAA"` |
| `+` | 1 or more repetitions | `r"A+"` | `"A"`, `"AA"`, `"AAA"` |
| `?` | 0 or 1 repetition | `r"AT?G"` | `"AG"` or `"ATG"` |
| `{m}` | Exactly *m* repetitions | `r"A{3}"` | `"AAA"` |
| `{m,n}` | Between *m* and *n* repetitions | `r"A{2,4}"` | `"AA"`, `"AAA"`, `"AAAA"` |
| `|` | Logical OR | `r"A|T"` | Matches `"A"` or `"T"` |
| `( )` | Grouping / capture | `r"(ATG)+"` | One or more `"ATG"` repeats |
| `\d` | Any digit | `r"\d+"` | `"123"`, `"007"` |
| `\D` | Any non-digit | `r"\D"` | `"A"`, `"_"` |
| `\w` | Any alphanumeric character | `r"\w+"` | `"A1B2"`, `"gene123"` |
| `\W` | Any non-alphanumeric | `r"\W+"` | `"@"`, `"#"`, `"%"` |
| `\s` | Whitespace (space, tab, newline) | `r"\s+"` | `" "`, `"\t"`, `"\n"` |
| `\S` | Non-whitespace | `r"\S+"` | `"ATGC"`, `"gene"` |

---

### Some useful tips
- Always prefix regex patterns with **`r""`** to make them *raw strings* and avoid issues with backslashes.  
  Example: `re.search(r"\d+", text)`
- Use `if re.search(pattern, text):` inside conditionals to control flow based on a match.  
- For multi-line FASTA or text data, use the `re.MULTILINE` flag when needed.  
- Use `re.compile()` for patterns reused inside loops — faster and cleaner.  
- Combine simple string methods and regex: sometimes `.split()` or `.replace()` is easier than a regex.


## Appendix: bioinformatics examples with regular expressions

Regular expressions (`re`) are incredibly powerful for parsing biological data.  
In bioinformatics, they’re often used to search for sequence motifs, identify FASTA headers, or extract specific information from messy text files.

Below are examples that build from simple to more realistic applications.

---

### 1. Detecting FASTA Headers

FASTA files start each sequence with a header line beginning with `>`.  
We can use `^>` (caret anchors to the *start* of a line) to find all headers.


In [8]:
import re

text = """>AY495386.1 Loxia curvirostra cytochrome b gene
ATGGCCCCAAATCTTCGTAAAAACCACC
>AY495387.1 Loxia leucoptera cytochrome b gene
ATGGCCCCTGATCTTCGTAAAAACCACC
"""

headers = re.findall(r"^>.*", text, re.MULTILINE)
print(headers)

['>AY495386.1 Loxia curvirostra cytochrome b gene', '>AY495387.1 Loxia leucoptera cytochrome b gene']


**Searching for DNA Triplets (Codons)**

This example uses the repetition operator {3} to match any 3-base codon.

In [10]:
seq = "ATGGCCCCAAATCTTCGTAAAAACCACCATTAGTGA"
codons = re.findall(r"[ATGC]{3}", seq)
print(codons[:10])  # print first 10 codons


['ATG', 'GCC', 'CCA', 'AAT', 'CTT', 'CGT', 'AAA', 'AAC', 'CAC', 'CAT']


**Identifying GC-Rich Regions**

Suppose you want to locate GC-rich subsequences — say, runs of at least six consecutive G or C bases.

In [11]:
seq = "ATGGCCCCAAATCTTCGTAAAAACCACCATTAGTGA"
gc_blocks = re.findall(r"[GC]{6,}", seq)
print("GC-rich blocks:", gc_blocks)

GC-rich blocks: ['GGCCCC']


**Conditional Example: Detecting Mitochondrial Genes**

In a loop that reads many headers, you could use regex conditionally to flag mitochondrial genes.

In [12]:
headers = [
    ">AY495386.1 Loxia curvirostra cytochrome b gene, mitochondrial",
    ">NC_001988.2 Homo sapiens COI gene, complete cds",
    ">AF345982.1 Gallus gallus nuclear gene"
]

for h in headers:
    if re.search(r"mitochondrial", h, re.IGNORECASE):
        print("Mitochondrial gene found:", h)


Mitochondrial gene found: >AY495386.1 Loxia curvirostra cytochrome b gene, mitochondrial
